# Project: Interconnect

## Introduction.

In the telecommunications industry, customer churn is a challenge that affects the stability and growth of businesses. Interconnect, a telecommunications operator, seeks to reduce churn by identifying at-risk customers early. To this end, the marketing team has collected data on customers and their contracts to develop a machine learning model that predicts the likelihood of churn.


## Goal.

Develop a machine learning model capable of predicting customer churn with high accuracy, using the AUC-ROC metric as the main performance indicator. This will allow Interconnect to implement targeted retention strategies, such as promotions and personalized plans, to reduce customer churn.

## Development plan

**1. Data Exploration and Preprocessing:** <br><br>

We will load and analyze the available datasets to understand their structure and detect potential issues. We will identify null values, duplicates, and inconsistencies that may affect the analysis. We will then evaluate how the datasets will be handled and merge them into a single table based on customerID. We will verify the data types and perform the necessary transformations to ensure their correct interpretation by the models. In addition, we will manage missing and outlier values, applying appropriate strategies to avoid bias in the modeling. <br>

**2. Exploratory Analysis:** <br><br>

We will explore the relationship between variables and the churn rate to understand which factors influence customer retention. We will identify relevant patterns and correlations that can provide useful information for prediction. We will also analyze the distribution of the target variable, checking for class balance and potential biases in the data.<br>

**3. Model Training and Evaluation:**<br><br>

We will select and train multiple classification models, including logistic regression, decision trees, and boosting-based models. We will compare the models' performance using AUC-ROC and accuracy as key metrics. Additionally, we will optimize the hyperparameters of the most promising model to improve its performance and achieve the best possible predictive capability.<br>

**4. Results and Conclusions:**<br><br>

We will analyze the selected model and evaluate its impact on churn prediction, ensuring it meets the project objectives. We will present recommendations based on the findings and discuss possible improvements. Finally, we will propose strategies for implementing the model within the company, enabling Interconnect to implement effective measures to retain its customers.<br>

## Development.

### 1. EDA and pre processing.

#### Library import.

In [ ]:

import lightgbm as lgb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


#### Load data

In [ ]:
try:
    contract_df = pd.read_csv("/Users/pauli/Documents/final_provider/contract.csv")
    internet_df = pd.read_csv("/Users/pauli/Documents/final_provider/internet.csv")
    personal_df = pd.read_csv("/Users/pauli/Documents/final_provider/personal.csv")
    phone_df = pd.read_csv("/Users/pauli/Documents/final_provider/phone.csv")

except FileNotFoundError:
    contract_df = pd.read_csv("/datasets/final_provider/contract.csv")
    internet_df = pd.read_csv("/datasets/final_provider/internet.csv")
    personal_df = pd.read_csv("/datasets/final_provider/personal.csv")
    phone_df = pd.read_csv("/datasets/final_provider/phone.csv")


#### Preprocessing data

##### Contract df

In [ ]:


contract_df.info()


The first thing we notice is that we need to change the column names to snake case and change the data type for the columns containing date information to datetime format. The rest of the data types appear to be correct. <br>
Something important is that the end_date column, which will be our target column, contains the string "No" to indicate the plans that are still active. Therefore, we'll create a new Boolean column to indicate the plans that have already ended and those that have not. We'll leave the end_date column as datetime so we can perform calculations if necessary.
Finally, we'll change the column type from total charges to numeric.

In [ ]:
# Rename columns

contract_df.columns = ['customer_id', 'begin_date', 'end_date', 'type', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges']

# Convert begin_date to datetime
contract_df['begin_date'] = pd.to_datetime(contract_df['begin_date'])

# Convert total_charges to numeric data.

contract_df["total_charges"] = pd.to_numeric(contract_df["total_charges"], errors="coerce").fillna(0.0)




In [ ]:
# Create a new column indicating whether the client is still active
contract_df['active_customer'] = (contract_df['end_date'] == 'No').astype(int)

# Convert end_date to datetime, keeping 'No' as is
contract_df['end_date'] = pd.to_datetime(contract_df['end_date'].replace('No', None))


print(contract_df.info())
print(contract_df.head())

In [ ]:
#Check for null or duplicate values

print (contract_df.isnull().sum())
print (contract_df.duplicated().sum())

It is observed that there are no duplicate values, and null values ??are only found in end_date as expected due to the change made above.

##### Internet DF

In [ ]:
internet_df.info()

The data types are correct, the only suggested change is to change the column names to snake type.

In [ ]:
internet_df.columns = ['customer_id', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies']

#Check name change

internet_df.head()

In [ ]:
#Check for null or duplicate values

print (internet_df.isnull().sum())
print (internet_df.duplicated().sum())

There are no null or duplicate values ??in this dataset.

##### Personal df

In [ ]:
personal_df.info()

The data types are correct, the only suggested change is to change the column names to snake type.

In [ ]:
# Rename
personal_df.columns = ['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents']

# Check name change

personal_df.head()

In [ ]:
#Check for null or duplicate values

print (personal_df.isnull().sum())
print (personal_df.duplicated().sum())

There are no null or duplicate values ??in this dataset.

##### Phone df

In [ ]:
phone_df.info()

The data types are correct, the only suggested change is to change the column names to snake type. 

In [ ]:
# Rename
phone_df.columns = ['customer_id', 'multiple_lines']

# Check name change

phone_df.head()


In [ ]:
#Comprobar si hay valores nulos o duplicados

print (phone_df.isnull().sum())
print (phone_df.duplicated().sum())

There are no null or duplicate values ??in this dataset.

#### Merge the Datasets

In [ ]:

merged_df = contract_df.merge(internet_df, on="customer_id", how="left")
merged_df = merged_df.merge(personal_df, on="customer_id", how="left")
merged_df = merged_df.merge(phone_df, on="customer_id", how="left")


merged_df.head()


Let's check the number of null values ??in each column of merged_df to see how complete the dataset is after the merge.

In [ ]:
merged_df.isnull().sum()


It appears that the columns with null values ??come from the internet_df dataset. Since this dataset contains information about internet services, the null values ??most likely correspond to customers who don't have internet service. Therefore, we'll replace these NaN values ??with 'No'. We'll do the same with the NaN values ??in multiple_lines.

In [ ]:
internet_cols = ["internet_service", "online_security", "online_backup", "device_protection", "tech_support", "streaming_tv", "streaming_movies"
]

merged_df[internet_cols] = merged_df[internet_cols].fillna("No")

merged_df["multiple_lines"] = merged_df["multiple_lines"].fillna("No")

In [ ]:
#Comprobar que ya no hay valores nulos

merged_df.isnull().sum()

In [ ]:
merged_df.info()


We cleaned and unified the data, ensuring consistency. We converted column names to snake_case, adjusted data types, and handled null values, replacing those for internet services and phone lines with "No." We also created the active_customer variable to identify active customers. We did not change categorical or binary variables to Boolean, as they will be coded later.

The final dataset has 7,043 records and 21 columns, ready for exploratory analysis.

### 2. EDA.

In [ ]:
sns.set_style("whitegrid")

# 1. Distribution of active vs. inactive customers
plt.figure(figsize=(6, 4))
sns.countplot(x="active_customer", data=merged_df)
plt.xticks([0, 1], ["Inactive", "Active"])
plt.xlabel("Customer Status")
plt.ylabel("Count")
plt.title("Distribution of Active vs. Inactive Customers")
plt.show()

The graph shows that there are significantly more active customers than inactive ones, indicating a class imbalance in the target variable. This can affect model performance, so we will later consider techniques such as oversampling, undersampling, or adjusting class weights to improve churn prediction.

In [ ]:
# Contract type distribution
plt.figure(figsize=(6, 4))
sns.countplot(y="type", data=merged_df, order=merged_df["type"].value_counts().index)
plt.xlabel("Count")
plt.ylabel("Contract Type")
plt.title("Distribution of Contract Types")
plt.show()

This chart shows that most customers have monthly contracts, while one- and two-year contracts are less common.
This could indicate that customers with monthly contracts have more flexibility to cancel service, which could be related to a higher churn rate. To confirm this, let's analyze the proportion of inactive customers with each contract type.

In [ ]:
contract_status_counts = contract_df.groupby(["type", "active_customer"]).size().unstack()

# Calculate the percentage of inactive customers by contract type
contract_status_counts["Inactive_Rate"] = contract_status_counts[0] / contract_status_counts.sum(axis=1)


plt.figure(figsize=(8, 5))
sns.barplot(
    x=contract_status_counts.index,
    y=contract_status_counts["Inactive_Rate"],
    palette=["#8da0cb"]
)
plt.ylabel("Inactive Customer Rate")
plt.xlabel("Contract Type")
plt.title("Proportion of Inactive Customers by Contract Type")
plt.ylim(0, 1)
plt.show()


The graph confirms what was expected: the cancellation rate is much higher for customers with monthly contracts compared to those with one- or two-year contracts.

In [ ]:
#Compare monthly and total charges between active and inactive customers

plt.figure(figsize=(12, 5))

# Monthly charges
plt.subplot(1, 2, 1)
sns.boxplot(x=contract_df["active_customer"], y=contract_df["monthly_charges"], palette=["#e41a1c", "#377eb8"])
plt.xlabel("Customer Status (0 = Inactive, 1 = Active)")
plt.ylabel("Monthly Charges ($)")
plt.title("Monthly Charges by Customer Status")

# Total charges
plt.subplot(1, 2, 2)
sns.boxplot(x=contract_df["active_customer"], y=contract_df["total_charges"], palette=["#e41a1c", "#377eb8"])
plt.xlabel("Customer Status (0 = Inactive, 1 = Active)")
plt.ylabel("Total Charges ($)")
plt.title("Total Charges by Customer Status")

plt.tight_layout()
plt.show()


**Monthly Fees:** Inactive customers (red) tend to pay higher monthly fees compared to active customers (blue). This suggests that customers with higher fees may cancel more frequently. <br>
**Total Fees:** Inactive customers have paid less overall, indicating that they cancel earlier. There are a large number of outliers among active customers, suggesting that some have been with the company for a long time and have accumulated high payments. This confirms the hypothesis that cancellations occur more frequently with short contracts or new customers.

Now, to streamline the model a bit, we'll eliminate columns that don't affect the prediction, starting with customer_id, since it's just an identifier, and total_charges, which is highly correlated with monthly_charges. We also eliminate the end_date, since it's only known for customers who have already left; it's not useful for predictions. We also have the column that indicates whether the customer is active or not.

In [ ]:
merged_df = merged_df.drop(columns=["customer_id", "total_charges", "end_date"])


Additionally, we're going to remove the begin date column and add one that calculates how many months the client has been around, to avoid formatting issues when arranging the data for the model.

In [ ]:

# Reference date
reference_date = pd.to_datetime("2020-02-01")

# Calculate customer tenure in full months
merged_df["months_active"] = ((reference_date - merged_df["begin_date"]).dt.days // 30).astype(int)

# Delete begin date
merged_df = merged_df.drop(columns=["begin_date"])



In [ ]:
merged_df.head()


In the EDA, we identified that most customers are still active, and that there is a significant imbalance between classes. We found that monthly contracts are associated with a higher churn rate. Furthermore, customers who cancel tend to have higher monthly charges but lower total spending, suggesting early cancellations.

We also analyzed temporal behavior and created a new variable, months_active, to represent customer tenure in months. Finally, we observed that numerical variables have moderate correlations with churn, indicating that categorical variables will play a key role in predictive models.

#### Prepare data for training and correct class imbalance with SMOTE.

In [ ]:

X = merged_df.drop(columns=['active_customer'])
y = merged_df['active_customer']

# OHE
X = pd.get_dummies(X, drop_first=True)


X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

# SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)



### 3. Model training and evaluation.

In [ ]:
#Sanity Test

dummy = DummyClassifier(strategy="most_frequent", random_state=15)
dummy.fit(X_train, y_train)

# Predict
y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)[:, 1]

# Evaluation
print("Dummy Classifier Report (Most Frequent Strategy):")
print(classification_report(y_test, y_pred_dummy))

roc_auc = roc_auc_score(y_test, y_proba_dummy)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dummy))


The model didn't predict any of them correctly. Although the accuracy is 73%, this is misleading: it only reflects that most customers are active. The model isn't learning anything useful.

This result is expected and desired as a baseline. Any model we test from now on must exceed these metrics.

In [ ]:
# Logistic Regression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(random_state=15, max_iter=1000)
logreg.fit(X_train_scaled, y_train_resampled)

y_pred_logreg = logreg.predict(X_test_scaled)
y_proba_logreg = logreg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_logreg))

roc_auc = roc_auc_score(y_test, y_proba_logreg)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_logreg))


The model correctly predicted 244 out of 374 churns, and correctly predicted 801 active customers, although it was mistaken for 234. It also achieved an overall accuracy of 74% and an ROC AUC of 0.80, indicating a good ability to distinguish between churning and remaining active customers.

In [ ]:
#RandomForestClassifier (default parameters)

rf = RandomForestClassifier(random_state=15)
rf.fit(X_train_resampled, y_train_resampled)


y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]


print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

roc_auc = roc_auc_score(y_test, y_proba_rf)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))


The model achieved an accuracy of 78%, which represents an improvement over the logistic regression model. It achieved an ROC AUC score of 0.8178, slightly exceeding the value obtained with logistic regression and suggesting a better ability to distinguish between customers who cancel and those who do not.

In [ ]:
#Random Forrest con Hiperpar?metros 


rf_base = RandomForestClassifier(random_state=15)


param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']}


grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=1)


grid_search.fit(X_train_resampled, y_train_resampled)


best_rf = grid_search.best_estimator_


y_pred_best = best_rf.predict(X_test)
y_proba_best = best_rf.predict_proba(X_test)[:, 1]

print("Best Random Forest Classification Report:")
print(classification_report(y_test, y_pred_best))

roc_auc = roc_auc_score(y_test, y_proba_best)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))


The model with tuned hyperparameters doesn't improve significantly compared to the baseline. The F1, ROC AUC, and accuracy are the same, so it's not worthwhile.

In [ ]:
#LightGBM 


lgbm = lgb.LGBMClassifier(random_state=15)
lgbm.fit(X_train_resampled, y_train_resampled)


y_pred_lgb = lgbm.predict(X_test)
y_proba_lgb = lgbm.predict_proba(X_test)[:, 1]


print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgb))

roc_auc = roc_auc_score(y_test, y_proba_lgb)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgb))


Although the churn F1 score is equal to that of Random Forest, the ROC AUC is the highest to date (0.82), indicating better overall discrimination between classes.
For class 0 (customers who canceled), it achieved a precision of 0.56 and a recall of 0.63, meaning it correctly detected 63% of actual churns.
In class 1 (active customers), the model maintained high performance with a precision of 0.86, a recall of 0.82, and an F1 score of 0.84, demonstrating good balance across both classes.

In [ ]:
#XG Boost

xgb = XGBClassifier(random_state=15, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train_resampled, y_train_resampled)


y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

roc_auc = roc_auc_score(y_test, y_proba_xgb)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))


It has an accuracy of 77%, on par with previous models. For class 0 (customers who canceled), it achieved a precision of 0.56, a recall of 0.61, and an F1-score of 0.58. In class 1 (active customers), XGBoost showed solid performance, with a precision of 0.85 and a recall of 0.82, resulting in an F1-score of 0.84, the same as LightGBM. The confusion matrix shows that the model correctly identified 227 customers who canceled and 853 who remained active.

The model achieved an ROC AUC score of 0.8205, which is lower than the 0.8250 obtained with LightGBM.

In [ ]:
#CatBoost

catboost = CatBoostClassifier(verbose=0, random_state=15)
catboost.fit(X_train_resampled, y_train_resampled)


y_pred_cb = catboost.predict(X_test)
y_proba_cb = catboost.predict_proba(X_test)[:, 1]

print("CatBoost Classification Report:")
print(classification_report(y_test, y_pred_cb))

roc_auc = roc_auc_score(y_test, y_proba_cb)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_cb))


CatBoost achieved an accuracy of 78%. For class 0, it achieved a precision of 0.57, a recall of 0.66, and an F1-score of 0.61, the highest value obtained so far for this class. This indicates that the model is more effective at identifying customers who actually cancel the service.

In class 1, performance was also solid, with an accuracy of 87% and an F1-score of 0.84. It also achieved an ROC AUC score of 0.8332, the best among all the models tested.

**These results position CatBoost as the best model so far, and since it stood out the most among the Boost-type models, we will try adding hyperparameters to see if it improves performance even further.**

In [ ]:
#CatBoost with Hyperparameters.


cat = CatBoostClassifier( verbose=0,random_state=15)


param_grid = {'depth': [4, 6, 8],
   'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [100, 300, 500],
    'l2_leaf_reg': [1, 3, 5]}

grid_search = GridSearchCV(
    estimator=cat,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=1
)


grid_search.fit(X_train_resampled, y_train_resampled)


best_cat = grid_search.best_estimator_


y_pred_best = best_cat.predict(X_test)
y_proba_best = best_cat.predict_proba(X_test)[:, 1]


print("Tuned CatBoost Classification Report:")
print(classification_report(y_test, y_pred_best))

roc_auc = roc_auc_score(y_test, y_proba_best)
print(f"ROC AUC Score: {roc_auc:.4f}")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

Hyperparameter tuning did not improve the model. Therefore, the original CatBoost remains the best model.

During the modeling stage, several classifiers were trained and evaluated with the aim of predicting customer churn. The logistic regression model served as a baseline, achieving decent performance with an ROC AUC of 0.8001. Subsequently, more complex models such as Random Forest, LightGBM, XGBoost, and CatBoost were evaluated, with CatBoost performing the best, with an F1-score of 0.61 for churn and an ROC AUC of 0.8332. Although attempts were made to tune the hyperparameters of some models, performance did not improve, confirming that the original CatBoost is an optimal fit for the available data and is the model chosen for this project.



## Results and conclusions


In the first phase of the project, four main data sources were worked with: contract information, internet services, personal data, and telephone services. The datasets were cleaned, renamed in snake_case format, and combined into a single main DataFrame. Date columns were transformed, a new variable called `months_active` representing customer tenure was generated, and null values ??and inconsistent data types were handled. Finally, `customer_id`, `total_charges`, and `end_date` were removed to reduce noise in the model. <br>

In the EDA, it was found that most customers were active, evidencing a class imbalance. Month-to-month contracts and payment methods such as e-check showed higher cancellation rates. Customers with high monthly charges and low tenure were more likely to cancel. Temporal, categorical, and numerical analyses were performed to detect patterns that could feed into the predictive model.

To prepare the model, categorical variables were encoded using one-hot encoding, and the predictor variables (X) were separated from the target variable (active_customer). The dataset was divided into training and test sets, and the SMOTE technique was applied to balance classes. For scale-sensitive models, StandardScaler was selectively applied. <br>

The models yielded the following results:

- **DummyClassifier**: served as a sanity test, with an ROC AUC of 0.50 and a recall of 0.00 for cancellations.
- **Logistic Regression**: ROC AUC of 0.8001 and F1-score of 0.57 for cancellations.
- **Random Forest**: improved performance with a ROC AUC of 0.8178 and F1-score of 0.60.
- **LightGBM**: achieved an ROC AUC of 0.8250, with an F1 score of 0.60.
- **XGBoost**: similar performance, with an ROC AUC of 0.8205 and an F1 score of 0.58.
- **CatBoost**: was the best model, with an ROC AUC of 0.8332 and an F1 score of 0.61. Hyperparameter tuning did not improve the results.

**Final Conclusion**
The CatBoost model, with its default parameters, established itself as the best option for predicting churn. Its balanced performance, especially in detecting churn, makes it ideal for integration into a retention strategy. It is recommended to implement it alongside a system that automates the sending of personalized promotions to at-risk users.
